# Human breast cancer protein preprocessing

Edit paths and dataset settings in `config.yaml` beside this notebook. Relative paths start from the experiment directory. Run cells from top to bottom.

Input: annotated IMC AnnData with `gaston_region`, `sample`, and 2D `spatial` coordinates. The original training used 25 measured protein channels after arcsinh(x/5) and channel-wise standard scaling. No learned embedding is fitted.


In [ ]:
from pathlib import Path
import sys
import yaml
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / "human_breast_cancer"
sys.path.insert(0, str(EXPERIMENT_DIR.parents[1] / "src"))
CONFIG = yaml.safe_load((EXPERIMENT_DIR / "config.yaml").read_text())
def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else EXPERIMENT_DIR / path


## Prepare protein features and save

Transform and scale measured protein channels, prepare coordinates, and save the training AnnData.


In [ ]:
import scanpy as sc
import numpy as np
from sklearn.preprocessing import StandardScaler
from stvirtual.data.protein import prepare_protein
from stvirtual.data.contracts import canonicalize_celltypes
adata = sc.read_h5ad(experiment_path(CONFIG["input_path"]))
for key in ["sample", CONFIG["celltype_source_key"]]:
    if key not in adata.obs or adata.obs[key].isna().any():
        raise ValueError(f"Missing or incomplete annotation: {key}")
adata.obs["sample"] = adata.obs["sample"].astype(str)
if CONFIG["protein_layer"] not in adata.layers:
    raw = adata.layers["raw"] if "raw" in adata.layers else adata.X
    raw = raw.toarray() if hasattr(raw, "toarray") else np.asarray(raw)
    adata.layers["raw"] = raw.copy()
    adata.layers["asinh"] = np.arcsinh(raw / 5.0)
    adata.layers[CONFIG["protein_layer"]] = StandardScaler().fit_transform(adata.layers["asinh"])
prepare_protein(adata, layer=CONFIG["protein_layer"], feature_key=CONFIG["latent_key"])
xy = np.asarray(adata.obsm["spatial"], dtype=np.float32)
if xy.shape != (adata.n_obs, 2) or not np.isfinite(xy).all():
    raise ValueError("IMC spatial coordinates must be finite and 2D")
adata.obs["cx_aligned"], adata.obs["cy_aligned"] = xy[:, 0], xy[:, 1]
mapping = canonicalize_celltypes(adata, source_key=CONFIG["celltype_source_key"])
output = experiment_path(CONFIG["prepared_path"])
output.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(output, compression="gzip")
mapping.to_csv(output.parent / "celltype_mapping.csv", index=False)
print(output, adata.obsm["X_protein"].shape)
